# NYU Deep Learning Midterm: Qwen 2B LoRA for Text-to-SVG

Fine-tunes `Qwen3.5-2B-Instruct` with LoRA on the competition's 50,000 training examples, then generates SVG submissions for the 1,000 test prompts.

**Competition constraints:**
- Valid XML with `<svg>` root element
- Maximum 16,000 characters
- Maximum 256 `<path>` elements
- Canvas: 256×256 pixels
- Allowed tags only: svg, g, path, rect, circle, ellipse, line, polyline, polygon, defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, title, desc, style, pattern, marker, filter

In [1]:
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Kaggle competition data paths
DATA_DIR = "/kaggle/input/competitions/dl-spring-2026-svg-generation"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "max_seq_length": 1024,
    "lora_r": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 24,
    "gradient_accumulation_steps": 8,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "eval_size": 0.02,
    "max_train_samples": 50_000,
    "max_new_tokens": 256,
    "inference_batch_size": 8,
    "output_dir": "/kaggle/working/qwen25_coder_svg_lora",
}

CONFIG

{'model_name': 'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit',
 'max_seq_length': 1024,
 'lora_r': 16,
 'lora_alpha': 16,
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 24,
 'gradient_accumulation_steps': 8,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'eval_size': 0.02,
 'max_train_samples': 50000,
 'max_new_tokens': 256,
 'inference_batch_size': 8,
 'output_dir': '/kaggle/working/qwen25_coder_svg_lora'}

In [4]:
# SVG constraint constants (from competition rules)
MAX_SVG_CHARS = 16_000
MAX_PATH_ELEMENTS = 256
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line",
    "polyline", "polygon", "defs", "use", "symbol", "clipPath",
    "mask", "linearGradient", "radialGradient", "stop", "text",
    "tspan", "title", "desc", "style", "pattern", "marker", "filter",
}


def _local_tag(element):
    """Strip XML namespace prefix from a tag, e.g. '{http://...}svg' -> 'svg'."""
    tag = element.tag
    return tag.split("}", 1)[-1] if "}" in tag else tag


def is_valid_svg(svg_text):
    """Return True if svg_text satisfies all competition constraints."""
    if not svg_text or not isinstance(svg_text, str):
        return False
    if len(svg_text) > MAX_SVG_CHARS:
        return False
    svg_text = svg_text.strip()
    if not svg_text.lower().startswith("<svg"):
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if _local_tag(root) != "svg":
        return False
    path_count = 0
    for el in root.iter():
        tag = _local_tag(el)
        if tag not in ALLOWED_TAGS:
            return False
        if tag == "path":
            path_count += 1
    if path_count > MAX_PATH_ELEMENTS:
        return False
    return True


def fallback_svg(_prompt=""):
    """Minimal valid 256x256 SVG used when generation fails."""
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" '
        'width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


# Quick sanity check
assert is_valid_svg(fallback_svg()), "Fallback SVG must be valid"
print("SVG validation helpers ready.")

SVG validation helpers ready.


In [5]:
# Load the competition training data
train_df = pd.read_csv(TRAIN_PATH)
print(f"Raw training rows: {len(train_df)}")
print(train_df.dtypes)
print(train_df.head(2))

Raw training rows: 50000
id        object
prompt    object
svg       object
dtype: object
                                 id  \
0  fd61e324e0cec5c11f337d0bfe2858c8   
1  999b3d4d5a860725bf9528910b5612f3   

                                              prompt  \
0  The image features two orange squares with a m...   
1  A simple smiley face with a wide open mouth an...   

                                                 svg  
0  <svg xmlns="http://www.w3.org/2000/svg" viewBo...  
1  <svg xmlns="http://www.w3.org/2000/svg" viewBo...  


In [6]:
# Filter training examples to only those with:
# - Non-empty prompt
# - SVG that satisfies all competition constraints
# (Trains the model to generate outputs that will actually score > 0)

train_df = train_df.dropna(subset=["prompt", "svg"])
train_df["prompt"] = train_df["prompt"].str.strip()
train_df["svg"] = train_df["svg"].str.strip()

# Filter: valid prompt
train_df = train_df[train_df["prompt"].str.len() > 0]

# Filter: SVG passes all competition constraints
print("Validating training SVGs (this may take a minute)...")
valid_mask = train_df["svg"].apply(is_valid_svg)
train_df = train_df[valid_mask].reset_index(drop=True)

print(f"Training rows after filtering: {len(train_df)}")
print(f"  Prompt length stats: {train_df['prompt'].str.len().describe().to_dict()}")
print(f"  SVG length stats:   {train_df['svg'].str.len().describe().to_dict()}")

Validating training SVGs (this may take a minute)...
Training rows after filtering: 49999
  Prompt length stats: {'count': 49999.0, 'mean': 116.5652513050261, 'std': 64.18806447975487, 'min': 5.0, '25%': 72.0, '50%': 103.0, '75%': 137.0, 'max': 860.0}
  SVG length stats:   {'count': 49999.0, 'mean': 2524.105362107242, 'std': 1773.4359089225525, 'min': 91.0, '25%': 1125.0, '50%': 2110.0, '75%': 3530.0, 'max': 15937.0}


In [7]:
# Convert to HuggingFace Dataset, cap samples, and split into train / eval
full_ds = Dataset.from_pandas(train_df[["prompt", "svg"]], preserve_index=False)
full_ds = full_ds.shuffle(seed=SEED)

if CONFIG["max_train_samples"] and len(full_ds) > CONFIG["max_train_samples"]:
    full_ds = full_ds.select(range(CONFIG["max_train_samples"]))
    print(f"Capped to {CONFIG['max_train_samples']} samples")

splits = full_ds.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows:  {len(eval_ds)}")
print("Sample:", train_ds[0]["prompt"][:80])

Train rows: 48999
Eval rows:  1000
Sample: A purple floppy disk with a red label slot.


In [8]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from a natural language description. "
    "Output only SVG code. The root element must be <svg> with width=\"256\" height=\"256\" "
    "viewBox=\"0 0 256 256\". Use only allowed tags. Keep the output under 16000 characters."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print("Example training text (first 400 chars):")
print(train_text[0]["text"][:400])

Map:   0%|          | 0/48999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Example training text (first 400 chars):
<|im_start|>system
You generate compact, valid SVG markup from a natural language description. Output only SVG code. The root element must be <svg> with width="256" height="256" viewBox="0 0 256 256". Use only allowed tags. Keep the output under 16000 characters.<|im_end|>
<|im_start|>user
A purple floppy disk with a red label slot.<|im_end|>
<|im_start|>assistant
<svg fill="none" height="128" vie


In [9]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print(model.print_trainable_parameters())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
None


In [10]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    args=training_args,
)

train_result = trainer.train()
print(train_result)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/48999 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/48999 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

...

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44,642 | Num Epochs = 1 | Total steps = 233
O^O/ \_/ \    Batch size per device = 24 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (24 x 8 x 1) = 192
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.413473,0.401276
200,0.380404,0.376611


TrainOutput(global_step=233, training_loss=0.44498374431430016, metrics={'train_runtime': 31597.1817, 'train_samples_per_second': 1.413, 'train_steps_per_second': 0.007, 'total_flos': 3.605707806793421e+17, 'train_loss': 0.44498374431430016, 'epoch': 1.0})


In [11]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

# Merge LoRA weights into the base model and save as 16-bit.
# Reloading with standard transformers (no Unsloth) bypasses the fused RoPE
# kernel that is incompatible with Qwen2.5's GQA architecture during inference.
# We reload in 4-bit via BitsAndBytes so VRAM usage stays the same.
merged_dir = CONFIG["output_dir"] + "_merged"
model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to: {merged_dir}")

Saved adapter + tokenizer to: /kaggle/working/qwen25_coder_svg_lora


config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:13<00:00, 13.86s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:18<00:00, 18.55s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qwen25_coder_svg_lora_merged`
Merged model saved to: /kaggle/working/qwen25_coder_svg_lora_merged
